In [16]:
import os
import time
import threading

#Target Files:
SAFE_FILE ="safe.txt"
SENSITIVE_FILE ="sensitive.txt"

#Vulnerable function
def write_to_file(filename, data):
    # CHECK:
    if os.access(filename, os.W_OK):
        print(f"[VICTIM] Passing the check. {filename} is writable.")


        time.sleep(0.1)  #Manually making a gap
        

        # USE:
        with open(filename, 'w') as f:
            f.write(data)
        print(f"[VICTIM] Wrote in: {filename}")
    else:
        print(f"[VICTIM] Access Denied: {filename}")


def attacker():
    time.sleep(0.05)  # TOC = 0  &  TOU = 0.1 so TOCTOU is in range (0 , 0.1) lets choose the middle: 0.05
    
    if os.path.exists(SAFE_FILE): #if safe.txt exists
        os.remove(SAFE_FILE) #remove it
        print("[ATTACKER] safe.txt removed")
        os.symlink(SENSITIVE_FILE, SAFE_FILE) #and make a symlink with name:safe.txt that shows the sensitive.txt
        print("[ATTACKER] symlink to sensitive.txt created")




#prepare the environment for simulating:
with open(SAFE_FILE, "w") as f:
    f.write("safe.txt data")
with open(SENSITIVE_FILE, "w") as f:
    f.write("sensitive.txt data.")


print("****Simulating TOCTOU Attack****")

#make a thread for attacker function. write_to_file function will be executed on the main thread at the same time:
t = threading.Thread(target=attacker)

#start the race:
t.start()
write_to_file(SAFE_FILE, "write_to_file function wrote the data here!")


t.join()

#RESULTS
print("\n\n****RESULT CONTROL****")
with open(SENSITIVE_FILE , "r") as f:
    print(f"Content of sensitive.txt: {f.read()}")


****Simulating TOCTOU Attack****
[VICTIM] Passing the check. safe.txt is writable.
[ATTACKER] safe.txt removed
[ATTACKER] symlink to sensitive.txt created
[VICTIM] Wrote in: safe.txt


****RESULT CONTROL****
Content of sensitive.txt: write_to_file function wrote the data here!
